# Лабораторная работа №5. Антифрод

**Цель:** обнаруживать мошеннические банковские операции (`Class=1`) и отличать их от легальных (`Class=0`).

Этот блокнот выполняет все вычислительные пункты задания:

1. Разделяет исходный набор данных на обучение (**70%**) и тест (**30%**) со стратификацией.
2. Обучает и настраивает **Bagging, Random Forest и LightGBM** и сравнивает их по **Macro F1**.
3. Настраивает **CatBoost**, сравнивает его с остальными моделями.
4. Строит **матрицу ошибок** и диаграмму сравнения моделей, сохраняет отчёт и результаты.

Четвёртый пункт исходного задания — **публикация кода в Git-репозитории** — выполняется отдельно через GitHub; инструкция есть в конце блокнота.

> Для запуска используй среду выполнения Colab с **CPU**. На Mac устанавливать Homebrew, `libomp` или Python-библиотеки не нужно.
>
> Нужен исходный файл **`creditcard.csv.zip`** из твоей лабораторной (или распакованный `creditcard.csv`).
> В блокноте нет заранее подставленных результатов: метрики вычисляются после реального обучения на загруженном файле.

## 1. Установка библиотек
Google Colab сам установит LightGBM и CatBoost. Если они уже доступны, повторная установка не нужна.

In [ ]:
import importlib.util
import subprocess
import sys

needed = [name for name in ("lightgbm", "catboost") if importlib.util.find_spec(name) is None]
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *needed])
print("Библиотеки LightGBM и CatBoost готовы")

## 2. Импорт библиотек
- `pandas`, `numpy` — обработка данных;
- `scikit-learn` — разделение выборки, Bagging, Random Forest и метрики;
- `LightGBM`, `CatBoost` — градиентный бустинг;
- `matplotlib` — графики.

In [ ]:
from pathlib import Path
import gc
import shutil
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 42
RESULTS_DIR = Path('/content/lab5_results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Папка результатов:', RESULTS_DIR)

## 3. Загрузка своего `creditcard.csv.zip`

При первом запуске Colab предложит выбрать файл на компьютере. Выбери **`creditcard.csv.zip`**, скачанный для лабораторной.

Если файл уже загружен в `/content`, повторно выбирать его не потребуется.

*ZIP не нужно распаковывать вручную.*

In [ ]:
from google.colab import files

folder = Path('/content')
options = [folder / 'creditcard.csv.zip', folder / 'creditcard.csv']
dataset_path = next((p for p in options if p.is_file()), None)

if dataset_path is None:
    print('Выбери на своём компьютере creditcard.csv.zip или creditcard.csv')
    uploaded = files.upload()
    candidates = [folder / name for name in uploaded]
    dataset_path = next((p for p in candidates if p.name.lower().endswith(('.csv.zip', '.csv', '.zip'))), None)

if dataset_path is None:
    raise FileNotFoundError('Не загружен CSV или ZIP с набором creditcard')

if dataset_path.suffix.lower() == '.zip':
    with zipfile.ZipFile(dataset_path) as archive:
        csv_names = [name for name in archive.namelist() if name.lower().endswith('.csv')]
        if not csv_names:
            raise ValueError('В архиве нет файла CSV')
        csv_name = next((name for name in csv_names if Path(name).name.lower() == 'creditcard.csv'), csv_names[0])
        with archive.open(csv_name) as f:
            df = pd.read_csv(f, dtype=np.float32)
else:
    df = pd.read_csv(dataset_path, dtype=np.float32)

df.columns = df.columns.str.strip()
if 'Class' not in df.columns:
    raise ValueError('Не найден столбец Class — проверь, что загружен нужный датасет')
if df.isna().any().any():
    raise ValueError('В данных обнаружены пропуски. Проверь исходный CSV')

df['Class'] = df['Class'].astype(np.int8)
print('Файл:', dataset_path.name)
print('Размер данных:', df.shape)
print('Количество объектов каждого класса:')
print(df['Class'].value_counts().sort_index())
display(df.head(3))

## 4. Деление на обучение, валидацию и тест

**Требование:** тестовая выборка — ровно 30%, при этом доля мошеннических операций сохраняется при помощи `stratify`.

Дополнительно выделяем из обучающей части валидационную выборку. На ней **выбираем параметры модели и порог предсказания**, а тест используем только для окончательной оценки. Это помогает избежать подгонки к тестовым данным.

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

X_train_all, X_test, y_train_all, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_all, y_train_all,
    test_size=0.20, stratify=y_train_all, random_state=RANDOM_STATE
)

print('Обучающая выборка:', X_train.shape[0], 'строк')
print('Валидационная выборка:', X_val.shape[0], 'строк')
print('Тестовая выборка:', X_test.shape[0], 'строк')
print('Мошенничество в тесте:', int(y_test.sum()))
print('Процент мошенничества в исходной выборке:', round(y.mean() * 100, 4))
print('Процент мошенничества в тесте:', round(y_test.mean() * 100, 4))

## 5. Функция оценки и подбор порога

`Macro F1` — среднее F1 по двум классам без учёта их размеров. Это важно: мошеннических операций намного меньше обычных.

Модели возвращают вероятность класса 1. Порог классификации подбираем **только на валидации**, после чего переносим его на тест без изменений.

In [ ]:
THRESHOLDS = np.array([
    0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.25,
    0.30, 0.35, 0.40, 0.45, 0.50, 0.55,
    0.60, 0.65, 0.70, 0.75, 0.80, 0.85,
    0.90, 0.95, 0.98, 0.99
])

def find_best_threshold(true_labels, probabilities):
    best_threshold = 0.5
    best_macro_f1 = -1.0
    for threshold in THRESHOLDS:
        predictions = (probabilities >= threshold).astype(int)
        score = f1_score(true_labels, predictions, average='macro', zero_division=0)
        if score > best_macro_f1:
            best_macro_f1 = score
            best_threshold = float(threshold)
    return best_threshold, best_macro_f1

def evaluate_model(model, X_eval, y_eval, threshold):
    probabilities = model.predict_proba(X_eval)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return {
        'Macro F1': f1_score(y_eval, predictions, average='macro', zero_division=0),
        'Precision (fraud)': precision_score(y_eval, predictions, zero_division=0),
        'Recall (fraud)': recall_score(y_eval, predictions, zero_division=0),
        'Accuracy': accuracy_score(y_eval, predictions),
    }, predictions

## 6. Модели и параметры для сравнения

Для каждого из четырёх алгоритмов рассматриваем **две настройки**, сравниваем Macro F1 на валидации и сохраняем лучшую настройку данного алгоритма.

- **Bagging**: несколько деревьев обучаются на случайных подвыборках;
- **Random Forest**: ансамбль случайных деревьев;
- **LightGBM**: градиентный бустинг деревьев;
- **CatBoost**: ещё одна реализация градиентного бустинга.

**Замечание:** запуск полного набора моделей требует ресурсов CPU. Не отключай среду Colab, пока идёт выполнение ячейки.

In [ ]:
def bagging_variant(n_estimators, max_samples):
    return BaggingClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=11, min_samples_leaf=4, random_state=RANDOM_STATE
        ),
        n_estimators=n_estimators,
        max_samples=max_samples,
        max_features=0.90,
        n_jobs=2,
        random_state=RANDOM_STATE
    )

candidates = {
    'Bagging': [
        ('20 деревьев', bagging_variant(20, 0.55)),
        ('40 деревьев', bagging_variant(40, 0.70)),
    ],
    'Random Forest': [
        ('100 деревьев', RandomForestClassifier(
            n_estimators=100, max_depth=12, min_samples_leaf=2,
            class_weight='balanced_subsample', n_jobs=2, random_state=RANDOM_STATE
        )),
        ('160 деревьев', RandomForestClassifier(
            n_estimators=160, max_depth=16, min_samples_leaf=2,
            class_weight='balanced_subsample', n_jobs=2, random_state=RANDOM_STATE
        )),
    ],
    'LightGBM': [
        ('180 итераций', LGBMClassifier(
            n_estimators=180, learning_rate=0.06, num_leaves=15,
            scale_pos_weight=10, verbose=-1, n_jobs=2, random_state=RANDOM_STATE
        )),
        ('300 итераций', LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=31,
            scale_pos_weight=10, verbose=-1, n_jobs=2, random_state=RANDOM_STATE
        )),
    ],
    'CatBoost': [
        ('240 итераций', CatBoostClassifier(
            iterations=240, depth=5, learning_rate=0.07,
            class_weights=[1, 10], loss_function='Logloss',
            random_seed=RANDOM_STATE, verbose=False,
            thread_count=2, allow_writing_files=False
        )),
        ('360 итераций', CatBoostClassifier(
            iterations=360, depth=6, learning_rate=0.06,
            class_weights=[1, 10], loss_function='Logloss',
            random_seed=RANDOM_STATE, verbose=False,
            thread_count=2, allow_writing_files=False
        )),
    ],
}

print('Модели готовы:', ', '.join(candidates))

## 7. Обучение и настройка моделей

Здесь действительно происходит обучение на загруженном датасете. Для каждого алгоритма выбираем вариант, показавший лучший Macro F1 на валидационной части.

In [ ]:
best_models = {}
validation_rows = []

for name, variants in candidates.items():
    print(f'\n=== {name} ===', flush=True)
    best_score = -1.0

    for variant_name, model in variants:
        model.fit(X_train, y_train)
        val_probabilities = model.predict_proba(X_val)[:, 1]
        threshold, score = find_best_threshold(y_val, val_probabilities)

        validation_rows.append({
            'Модель': name,
            'Вариант': variant_name,
            'Порог': threshold,
            'Macro F1 (validation)': score,
        })
        print(f'  {variant_name}: Macro F1={score:.5f}, порог={threshold:.2f}', flush=True)

        if score > best_score:
            best_score = score
            best_models[name] = {
                'model': model,
                'threshold': threshold,
                'variant': variant_name,
            }
        else:
            del model
        gc.collect()

validation_table = pd.DataFrame(validation_rows).sort_values(
    'Macro F1 (validation)', ascending=False
)
display(validation_table)
validation_table.to_csv(RESULTS_DIR / 'validation_tuning.csv', index=False)
print('Подбор гиперпараметров завершён')

## 8. Окончательная оценка на тесте

Тестовая выборка ещё не использовалась для выбора моделей или порогов. Сейчас для каждого алгоритма рассчитываем метрики именно на тесте.

**Лучшая модель** определяется по максимальному `Macro F1` на тестовой выборке среди заранее выбранных вариантов каждого алгоритма. Сравнение моделей по тесту следует интерпретировать осторожно: оно не заменяет независимую внешнюю проверку.

In [ ]:
test_rows = []
test_predictions = {}

for name, info in best_models.items():
    metrics, predictions = evaluate_model(
        info['model'], X_test, y_test, info['threshold']
    )
    test_predictions[name] = predictions
    test_rows.append({
        'Модель': name,
        'Выбранная настройка': info['variant'],
        'Порог': info['threshold'],
        **metrics,
    })

results = pd.DataFrame(test_rows).sort_values('Macro F1', ascending=False).reset_index(drop=True)
display(results.round(5))
results.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
best_name = results.loc[0, 'Модель']
print('Лучшая модель по Macro F1 на тесте:', best_name)
print('Macro F1:', round(float(results.loc[0, 'Macro F1']), 5))

## 9. График сравнения Macro F1 и матрица ошибок

Матрица ошибок:
- **TN** — обычные операции, верно определённые как обычные;
- **FP** — обычные операции, ошибочно определённые как мошеннические;
- **FN** — пропущенное мошенничество;
- **TP** — правильно обнаруженное мошенничество.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(results['Модель'], results['Macro F1'])
ax.set_ylabel('Macro F1')
ax.set_title('Лабораторная №5: сравнение моделей на тесте')
ax.set_ylim(0, 1)
for index, score in enumerate(results['Macro F1']):
    ax.text(index, score + 0.015, f'{score:.4f}', ha='center')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'macro_f1_comparison.png', dpi=160)
plt.show()

cm = confusion_matrix(y_test, test_predictions[best_name], labels=[0, 1])
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Легальная (0)', 'Мошенническая (1)']
).plot(ax=ax, values_format='d', colorbar=False)
ax.set_title(f'Матрица ошибок: {best_name}')
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'confusion_best.png', dpi=160)
plt.show()

TN, FP, FN, TP = cm.ravel()
print(f'TN={TN}, FP={FP}, FN={FN}, TP={TP}')
print(f'Мошеннических транзакций обнаружено: {TP} из {TP + FN}')

## 10. Создание отчёта и архива результатов

Сохраняем таблицы в CSV, графики в PNG, а пояснение — в Markdown. Готовый ZIP автоматически предложит скачать браузер.

In [ ]:
report_lines = [
    '# Лабораторная №5. Результаты обнаружения мошенничества',
    '',
    f'Всего наблюдений: {len(df):,}.',
    f'Обычных транзакций: {int((y == 0).sum()):,}.',
    f'Мошеннических транзакций: {int((y == 1).sum()):,}.',
    f'Тестовая выборка: {len(y_test):,} строк (30%, stratify).',
    '',
    '## Сравнение моделей на тесте',
    '',
    '| Модель | Macro F1 | Precision (мошенничество) | Recall (мошенничество) |',
    '|---|---:|---:|---:|',
]
for _, row in results.iterrows():
    report_lines.append(
        f"| {row['Модель']} | {row['Macro F1']:.5f} | "
        f"{row['Precision (fraud)']:.5f} | {row['Recall (fraud)']:.5f} |"
    )
report_lines += [
    '',
    f'Лучшая модель на тесте: **{best_name}**.',
    '',
    '## Матрица ошибок',
    '',
    f'- TN = {TN}',
    f'- FP = {FP}',
    f'- FN = {FN}',
    f'- TP = {TP}',
    '',
    'Подбор гиперпараметров и порога выполнялся на валидации, без использования теста.',
    'Результаты тестирования носят экспериментальный характер.',
]
(RESULTS_DIR / 'RESULTS_REPORT.md').write_text('\n'.join(report_lines), encoding='utf-8')

zip_path = shutil.make_archive('/content/lab5_results', 'zip', RESULTS_DIR)
print('Отчёт и изображения сохранены в:', RESULTS_DIR)
print('Архив готов:', zip_path)
from google.colab import files
files.download(zip_path)

## 11. Что показывать преподавателю

1. Ячейку **разбиения выборки**: `test_size=0.3`, `stratify=y`, `random_state=42`.
2. Ячейки **обучения**: Bagging, Random Forest, LightGBM и CatBoost, два набора гиперпараметров для каждого.
3. Таблицу **Macro F1**, объяснить, почему `accuracy` недостаточно при крайне редком мошенничестве.
4. **Матрицу ошибок** лучшего классификатора, пояснить TN, FP, FN и TP.
5. Файлы `RESULTS_REPORT.md`, `model_comparison.csv`, `confusion_best.png`.

### Как выполнить пункт «код выложить в Git-репозиторий»

Создай новый репозиторий на [github.com](https://github.com/new) с названием `lab5-antifraud` и загрузи через **Add file → Upload files** этот ноутбук `.ipynb` и `README.md` из подготовленного архива. Большой датасет `creditcard.csv.zip` публиковать не нужно: он доступен по ссылке из задания.

**Для защиты:** Bagging — ансамбль на случайных подвыборках, Random Forest — случайный лес, LightGBM и CatBoost — градиентный бустинг. Macro F1 — среднее гармонических оценок F1 по двум классам.